Randomised Riemannian Hamiltonian Monte Carlo for Sampling on $S^{5} \subset \mathbb{R}^{6}$

Defined by $g(x) = \sum^{6}_{i=1}x^{2}_{i} - 1 = 0$, we have $\mathcal{M} := \{x \in \mathbb{R}^{6} \mid g(x) = 0\}$

$T_{p}\mathcal{M} =  \{v \mid \langle \nabla g(p),v \rangle = 0 \}$

$G(q) = g'(q) = (2x_{i})^{6}_{i=1}$

$\pi = I - G^T (G G^T)^{-1} G = I - G^T G/4$ is the projection onto the tangent space.

1) Dynamics

Simulating Hamiltonian Dynamics with potential governed by
$U(q) = -\log{\pi_{\mathcal{H}}(q)}$, where $\pi_{\mathcal{H}}$ is the density function of the Bingham-von Mises-Fisher distribution. To compare 3 different sampling methods.

The Bingham-von Mises-Fisher distribution on the Sphere is determined by

$\pi_{\mathcal{H}}(x) \propto \exp{\{ c^{T}x + x^{T}Ax\}}$
in $\mathbb{R}^{3}$, $c \propto \mu$



We have $\nabla_{q} V(q) = -\nabla_{q}\log{\pi_{\mathcal{H}}(q)} = -\nabla_{q}(c^{T}q+q^{T}Aq) = -c^{T} - 2Aq$

In [ ]:
import numba
import numpy as np
from numpy.linalg import inv
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import time

#Spherical Coefficients

#Bi Modal
c1 = 40.
mean = np.array([c1,0.,0.,0.,0.])
A = np.diag(np.array([-20.,-10.,0.,10.,20.]))
dimension  = 5


'Numba functions'

@numba.jit(nopython=True)
def matrix_vec_multiplication(A,x):
    v = np.zeros(len(A))
    
    for i in range(len(A)):
        for j in range(len(x)):
                v[i] += A[i][j] * x[j]
    return v

@numba.jit(nopython=True)
def dot_product(v1,v2):
    dot = 0
    for i in range(len(v1)):
        
        dot += v1[i]*v2[i]
        
    return dot

@numba.jit(nopython=True)
def g(q):
    #.sum
    y = np.sum(np.power(q,2)) - 1
    return y


@numba.jit(nopython=True)
def G(q):
    z = 2*q
    return z

#need to do this onwards
@numba.jit(nopython=True)
def potential_derv(q):
    v = matrix_vec_multiplication(A,q)
    mu = - mean - 2*v
    return mu
        

In [ ]:
#RATTLE Hamiltonian Flow
@numba.jit(nopython=True)
def RATTLE_with_Potential(x0,v0,t,dt):
    n = np.floor(t/dt)
    vn = v0
    qn = x0
    vhalf = v0
    G_q = G(qn)
    pderv = potential_derv(qn)
    for i in range(int(n)):
        
        
        
        #solver for Lagrange position multipliers
        Q = qn + vn*dt - 0.5*dt*dt*pderv
        for j in range(50):
            g_Q = g(Q)
            dlambda = g_Q/dot_product(G(Q),G_q)
            Q = Q - G_q*dlambda
            if abs(g_Q) < 1e-8:
                break
        
        #half step
        vhalf = (Q-qn)/dt
        qn = Q
    
        
        pderv =  potential_derv(qn)
        G_q = G(qn)
        gram_inv = 1/dot_product(G_q,G_q)
        
        #linear solver Lagrange velocity multipliers
        b = np.dot(G_q,2*vhalf/dt - pderv)
        coeffs_v = gram_inv*b
        
        #full step
        vn = vhalf - 0.5*dt*pderv - 0.5*dt*G(qn)*coeffs_v

        
            
    return qn,vn

In [ ]:
import time
v1 = np.array([1.0,2.0,3.0,4.0,5.0])
x = np.array([0.0,0.0,0.0,0.0,1.0])
v = np.array([1.0,1.0,0.0,0.0,0.0])
t = 1000
dt = 0.01
print(matrix_vec_multiplication(A,v1))
print(potential_derv(np.array(v1)))
print(dot_product(v1,v1))
t1 = time.time()
print(RATTLE_with_Potential(x,v,t,dt))
t2 = time.time()
print('time when compiling',t2-t1)

t3 = time.time()
print(RATTLE_with_Potential(x,v,t,dt))
t4 = time.time()
print('numba time=',t4-t3)

2) Event Time Sampling


In [ ]:
#Sampling Event Times
@numba.jit(nopython=True)
def time_exp(lam):
    t = np.random.exponential(lam)
    return t

3) Gaussian Sampling on Tangent Space

In [ ]:
@numba.jit(nopython=True)
def tangent_space_gaussian(q):
    
    v = np.random.normal(0.,1.0,dimension).T
   
    G_q = G(q)
    
    gram_inv = 1/dot_product(G_q,G_q)
    
    proj_matrix = np.eye(dimension) - gram_inv*np.outer(G_q,G_q)
    
    #sample 3d gaussian and then project onto tangent space.
    v = matrix_vec_multiplication(proj_matrix,v)
    
    return v
print(tangent_space_gaussian(np.array([0.,0.,0.,0.,-1.])))

4) Simulation

In [ ]:
@numba.jit(nopython=True)
def hamiltonian(x,v):
    h = -(dot_product(mean,x) + dot_product(x,matrix_vec_multiplication(A,x)))+ 0.5*np.linalg.norm(v)**2
    return h

In [ ]:
x = np.array([0.,0.,0.,0.,-1.])
v = tangent_space_gaussian(x)
hamiltonian(x,v)

In [ ]:
@numba.jit(nopython=True)
def RRHMC(num_of_events,dt_max,T,x_init):
    
    #Exponential Expected Value
    rate = T
    x = x_init
    
    position_list = [x_init]
    v = tangent_space_gaussian(x)
    
    accept = 0.
    gradient_evaluations = 0
    
    for i in range(num_of_events):
        
        t = time_exp(rate)
        L = np.ceil(t/dt_max)
        
        gradient_evaluations += L
        dt = t/L
        
        h = hamiltonian(x,v)
        
        xnew,vnew = RATTLE_with_Potential(x,v,t,dt)
        
        h_new = hamiltonian(xnew,vnew) 
        
        #metropolis hasting step
        u = np.random.rand()
        α = np.exp(-h_new+h)
        accept += min(1,α)
        if u <= α:
            x = xnew
         
        position_list.append(x)
        
        #resampling velocity
        v = tangent_space_gaussian(x)
        if i%10000 == 0:
            print(i)
    
    accept/=num_of_events
    gradient_evaluations /= num_of_events
            
    return position_list,accept,gradient_evaluations
    
    

In [ ]:
@numba.jit(nopython=True)
def RHMC(num_of_events,dt,T,x_init):
    
    #Exponential Expected Value
    x = x_init
    
    position_list = [x_init]
    v = tangent_space_gaussian(x)
    
    
    accept = 0.
    
    for i in range(num_of_events):
        
        h = hamiltonian(x,v)
        
        xnew,vnew = RATTLE_with_Potential(x,v,T,dt)
        
        h_new = hamiltonian(xnew,vnew) 
        
        #metropolis hasting step
        u = np.random.rand()
        α = np.exp(-h_new+h)
        accept += min(1,α)
        if u <= α:
            x = xnew
        
    
        position_list.append(x)
        
        #resampling velocity
        v = tangent_space_gaussian(x)
        if i%10000 == 0:
            print(i)
  
    accept/=num_of_events
    
    return position_list,accept


In [ ]:
x_init = np.array([0.,0.,0.,0.,1.])
dt = 0.05
T = 0.1
DT_samples,accept_DT = RHMC(1000,dt,T,x_init)
RT_samples,accept_RT,gradient_evaluations = RRHMC(1000,dt,T,x_init)


# Bias at different step sizes.

In [ ]:
num_of_events = 100
x_init = np.array([0,0,1])
λ = 0.5
@numba.jit(nopython=True)
def f(q):
    return -dot_product(mean,q) - dot_product(q,matrix_vec_multiplication(A,q))

## Mean estimate

In [ ]:
x_init = np.array([0.,0.,0.,0.,1.])
for dt_ in [0.1,0.01,0.001]:
    RT_samples,accept_RT,gradient_evaluations = RRHMC(num_of_events,dt_,λ,x_init)
    RT_means = []
    
    
    counter = 0
    mean_curr = 0.
    for i in RT_samples:

        
        mean_curr += f(i)
        mean_current = mean_curr/(counter + 1)
        
        RT_means.append(mean_current)
    
        counter += 1
   
    mean_estimate_RT = RT_means[-1]
    print('RT mean Estimate =', mean_estimate_RT)
    plt.plot(RT_means,label='dt = %.5f' %dt_)
#plt.plot(np.ones(num_of_events)*mean_estimate, label = 'True Value')
plt.legend()
plt.title('Randomised Time')    
plt.show()



In [ ]:
x_init = np.array([0.,0.,0.,0.,1.])

for dt_ in [0.1,0.01,0.001]:
    mean_list = []
    DT_samples,accept_DT = RHMC(num_of_events,dt_,λ,x_init)

    DT_means = []
    counter = 0
    mean_curr = 0.
    for i in DT_samples:

        
        mean_curr += f(i)
        mean_current = mean_curr/(counter + 1)
        
        DT_means.append(mean_current)
    
        counter += 1
   
    mean_estimate_DT = DT_means[-1]
    print('DT mean Estimate =', mean_estimate_DT)
    plt.plot(DT_means,label='dt = %.5f' %dt_)

#plt.plot(np.ones(num_of_events)*mean_estimate, label = 'True Value')
plt.legend()
plt.title('Deterministic Time')    
plt.show()

In [ ]:
plt.plot(DT_means,label='DT')
plt.plot(RT_means,label='RT')
#plt.plot(np.ones(num_of_events)*mean_estimate,'--',label = 'True Value')
plt.title('Monte Carlo Average of $-\log \pi$')
plt.legend()

In [ ]:
EX = -27.867

# Loop

In [ ]:
@numba.jit(nopython=True)
def IAC(f_list,lag,EX):
    N = len(f_list)
    AC_list = np.zeros(lag)
    for i in range(lag):
        for j in range(N-i):
            AC_list[i] += (f_list[j]-EX)*(f_list[j+i]-EX)/(N-i)
    return np.sum(AC_list)/AC_list[0]


In [ ]:
import statsmodels.api as sm
T_list = np.linspace(0.001,1.0,500)
print(T_list)
number = len(T_list)
RMHMC_list = []
RT_RMHMC_list = []
c = np.array(mean)


def d(x,y):
    return (np.arccos(np.dot(x,y)))**2



def f(q):
    
    return -np.dot(c,q) - np.dot(q,np.matmul(A,q))



for i in T_list:
    #Initialise
    T = i
    num_of_events = 100000
    dt = 0.001
    print('WE ARE AT STAGE =',i)

    #Random initialisation of x on S^5
    x_init = np.array([0.,0.,0.,0.,1.])
    
    #Samples
    pos,accept_RT, gradient_evaluations = RRHMC(num_of_events,dt,T,x_init)
        
    GMC_samples,accept_DT = RHMC(num_of_events,dt,T,x_init)
    
    
    
    #IAC GMC
    front = len(GMC_samples)//10
    N = len(GMC_samples)- front

    auto_corr_vector_GMC = []
    for i in GMC_samples:
        auto_corr_vector_GMC.append(f(i))
    
    IAC_GM = IAC(np.array(auto_corr_vector_GMC[front:]),N//50,EX)
    
    RMHMC_list.append(IAC_GM)
    
    #IAC RT-RMHMC
    front_RM = len(pos)//10

    N = len(pos)-front_RM
    auto_corr_vector_RRHMC = []

    for i in pos:
           auto_corr_vector_RRHMC.append(f(i))
    
    IAC_RT = IAC(np.array(auto_corr_vector_RRHMC[front_RM:]),N//50,EX)
    
    
    RT_RMHMC_list.append(IAC_RT)
    
    print('===================================================================')
    print(RT_RMHMC_list)
    print('===================================================================')
    
    print('===================================================================')
    print(RMHMC_list)
    print('===================================================================')
    
    plt.title('IAC for varying $\lambda$')
    plt.plot(np.array(T_list)[:len(RT_RMHMC_list)],2*np.array(RT_RMHMC_list)-1,'x-',label='RT-RMHMC')
    plt.plot(np.array(T_list)[:len(RT_RMHMC_list)],2*np.array(RMHMC_list)-1,'x-',label='RMHMC')
    plt.plot(np.array(T_list)[:len(RT_RMHMC_list)], np.ones(len(T_list[:len(RT_RMHMC_list)])),'--')
    plt.xlabel('Mean duration $\lambda$')
    plt.ylabel('IAC')
    plt.yscale('log')
    plt.grid()
    plt.legend()
    plt.show()
    
    

In [ ]:
plt.title('IAC for varying $\lambda$')
plt.plot(np.array(T_list)[:len(RT_RMHMC_list)],2*np.array(RT_RMHMC_list)-1,'x-',label='RT-RMHMC')
plt.plot(np.array(T_list)[:len(RT_RMHMC_list)],2*np.array(RMHMC_list)-1,'x-',label='RMHMC')
plt.plot(np.array(T_list)[:len(RT_RMHMC_list)], np.ones(len(T_list[:len(RT_RMHMC_list)])),'--')
plt.xlabel('Mean duration $\lambda$')
plt.ylabel('IAC')
plt.yscale('log')
plt.grid()
plt.legend()
plt.show()